# Week 3 Task: Unsupervised Learning and Clustering Analysis
## Segmentation of Wholesale Customer Purchasing Behaviors

**Author:** Data Science & Machine Learning Analytics Team  
**Date:** August 2026  
**Tech Stack:** Python 3.10, Scikit-Learn, Pandas, NumPy, Matplotlib, Seaborn  

--- 
### Objective
Apply unsupervised learning algorithms (K-Means and Hierarchical Agglomerative Clustering) to segment wholesale customers based on annual spending across six product categories: Fresh, Milk, Grocery, Frozen, Detergents_Paper, and Delicassen.

In [ ]:
# Step 1: Library Imports & Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage

sns.set_theme(style='whitegrid')
np.random.seed(42)

In [ ]:
# Step 2: Ingest / Load Dataset
# For reproduction, we read the generated dataset wholesale_customers_data.csv
df = pd.read_csv('wholesale_customers_data.csv')
print('Dataset Shape:', df.shape)
df.head()

In [ ]:
# Step 3: Exploratory Data Analysis & Summary Statistics
print('Missing Values:\n', df.isnull().sum())
df.describe().T

In [ ]:
# Step 4: Data Preprocessing & Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)
print('Scaled Feature Matrix Shape:', X_scaled.shape)

In [ ]:
# Step 5: Optimal Cluster Selection (Elbow Method & Silhouette Scores)
k_range = range(2, 9)
wcss, sil_scores, ch_scores, db_scores = [], [], [], []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    wcss.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))
    ch_scores.append(calinski_harabasz_score(X_scaled, labels))
    db_scores.append(davies_bouldin_score(X_scaled, labels))

# Visualization
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(list(k_range), wcss, 'o-', color='#1f77b4', linewidth=2)
ax[0].set_title('Elbow Method (WCSS vs K)', fontweight='bold')
ax[0].set_xlabel('Clusters (K)')
ax[0].set_ylabel('WCSS')

ax[1].plot(list(k_range), sil_scores, 's--', color='#2ca02c', linewidth=2)
ax[1].axvline(x=4, color='red', linestyle=':', label='Optimal K=4')
ax[1].set_title('Silhouette Score vs K', fontweight='bold')
ax[1].set_xlabel('Clusters (K)')
ax[1].set_ylabel('Silhouette Score')
ax[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Step 6: Model Training (K-Means & Hierarchical Agglomerative)
km_4 = KMeans(n_clusters=4, random_state=42, n_init=10)
df['KMeans_Cluster'] = km_4.fit_predict(X_scaled)

agg_4 = AgglomerativeClustering(n_clusters=4, linkage='ward')
df['Hierarchical_Cluster'] = agg_4.fit_predict(X_scaled)

print('K-Means Cluster Counts:\n', df['KMeans_Cluster'].value_counts())
print('Hierarchical Cluster Counts:\n', df['Hierarchical_Cluster'].value_counts())

In [ ]:
# Step 7: Hierarchical Dendrogram Visualization
plt.figure(figsize=(10, 5))
linked = linkage(X_scaled, method='ward')
dendrogram(linked, truncate_mode='lastp', p=30, leaf_rotation=45, show_contracted=True)
plt.title('Hierarchical Clustering Dendrogram (Ward Linkage)', fontweight='bold')
plt.xlabel('Sample Index / Cluster Size')
plt.ylabel('Euclidean Distance')
plt.axhline(y=18, color='r', linestyle='--', label='Cut Threshold (K=4)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Step 8: PCA Dimensionality Reduction & Scatter Visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
df['PCA1'] = X_pca[:, 0]
df['PCA2'] = X_pca[:, 1]

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=df, x='PCA1', y='PCA2', hue='KMeans_Cluster', palette='Set1', ax=ax[0], s=60, alpha=0.8)
ax[0].set_title('K-Means Clusters (PCA Space)', fontweight='bold')
sns.scatterplot(data=df, x='PCA1', y='PCA2', hue='Hierarchical_Cluster', palette='Set2', ax=ax[1], s=60, alpha=0.8)
ax[1].set_title('Hierarchical Clusters (PCA Space)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Step 9: Cluster Characteristic Profiling
feature_cols = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
profile = df.groupby('KMeans_Cluster')[feature_cols].mean()
print('Cluster Centroids (Mean Spending):')
display(profile)

profile.T.plot(kind='bar', figsize=(10, 5), colormap='viridis', width=0.8)
plt.title('Mean Spending by Product Category per Cluster', fontweight='bold')
plt.ylabel('Mean Annual Spend (INR/USD)')
plt.legend(title='Cluster', labels=['C0: Retail/Grocery', 'C1: HORECA/Fresh', 'C2: Supermarket', 'C3: Small/Budget'])
plt.tight_layout()
plt.show()